# Preprocessing Configuration Demo — Airline Sentiment Clustering

This Colab-ready notebook demonstrates the repository's modular P0, P1 and P2 preprocessing configurations and several controlled variants.

**Scope disclaimer:** these preprocessing choices were designed and audited for the airline-tweet sentiment-clustering context. Re-audit corpus-specific assumptions before reuse in another domain.

The raw `text` column is always retained.

## 1. Connect Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Download / update the project code from GitHub

In [2]:
import os

REPO_URL = "https://github.com/frankie-chew-nus/sentiment-clustering-ml.git"
REPO_DIR = "/content/sentiment-clustering-ml"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git switch main
    !git pull origin main

%cd {REPO_DIR}
!git branch --show-current
!git log -1 --oneline

Cloning into '/content/sentiment-clustering-ml'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 376 (delta 46), reused 0 (delta 0), pack-reused 239 (from 2)
Receiving objects: 100% (376/376), 150.03 KiB | 3.26 MiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/sentiment-clustering-ml
main
a9a557f (HEAD -> main, origin/main, origin/HEAD) Merge pull request #22 from frankie-chew-nus/feature/l3-master-feature-matrix


## 3. Install dependencies

In [3]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q -r requirements-dev.txt
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by sel

## 4. Load preprocessing code and configurations

In [4]:
import sys, importlib
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import config, lexical_normalization, preprocessing
importlib.reload(config)
importlib.reload(lexical_normalization)
importlib.reload(preprocessing)

from config import P0_CONFIG, P1_CONFIG, P2_CONFIG
from preprocessing import preprocess_text

## 5. Load the raw CSV from Google Drive

Edit `INPUT_CSV` if your Drive path differs.

In [5]:
import pandas as pd
from pathlib import Path

INPUT_CSV = "/content/drive/My Drive/MSI5001 project/EDA/Tweets-redacted.csv"
assert Path(INPUT_CSV).exists(), f"File not found: {INPUT_CSV}"

df = pd.read_csv(INPUT_CSV)
print("Rows:", len(df))
print("Columns:", list(df.columns))
display(df[["text"]].head())

Rows: 14639
Columns: ['text']


,text
0,@VirginAmerica What @dhepburn said.
1,@VirginAmerica plus you've added commercials t...
2,@VirginAmerica I didn't today... Must mean I n...
3,@VirginAmerica it's really aggressive to blast...
4,@VirginAmerica and it's a really big bad thing...


## 6. Compare P0 / P1 / P2 on representative tweets

In [6]:
examples = [
    "@AmericanAir I won't fly American Airlines again!!! 😡",
    "This is #badcustomerservice",
    "Delayed &amp; nobody is helping",
    "The flights were delayed and I am very disappointed!!!",
    "The service was really very good 😊",
    "10/10 service 😊",
    "-9/10 experience",
]

for text in examples:
    print("RAW:", text)
    print("P0 :", preprocess_text(text, P0_CONFIG))
    print("P1 :", preprocess_text(text, P1_CONFIG))
    print("P2 :", preprocess_text(text, P2_CONFIG))
    print("-" * 90)

RAW: @AmericanAir I won't fly American Airlines again!!! 😡
P0 : airline I won't fly airline again!!! 😡
P1 : airline i will not fly airline again!!! enraged face
P2 : airline not fly airline ! ! ! enraged face
------------------------------------------------------------------------------------------
RAW: This is #badcustomerservice
P0 : This is badcustomerservice
P1 : this is badcustomerservice
P2 : bad customer service
------------------------------------------------------------------------------------------
RAW: Delayed &amp; nobody is helping
P0 : Delayed & nobody is helping
P1 : delayed & nobody is helping
P2 : delay & nobody help
------------------------------------------------------------------------------------------
RAW: The flights were delayed and I am very disappointed!!!
P0 : The flights were delayed and I am very disappointed!!!
P1 : the flights we are delayed and i am very disappointed!!!
P2 : flight delay very disappointed ! ! !
-------------------------------------------

## 7. Define controlled configuration variants

In [7]:
P2_KEEP_STOPWORDS = {**P2_CONFIG, "remove_stopwords": False}
P2_NO_LEMMA = {**P2_CONFIG, "lemmatize": False}
P2_NO_HASHTAG_SPLIT = {**P2_CONFIG, "hashtag_segmentation_mode": "none"}
P1_PLUS_LEMMA = {**P1_CONFIG, "lemmatize": True, "spacy_model": "en_core_web_sm"}

VARIANTS = {
    "P0": P0_CONFIG,
    "P1": P1_CONFIG,
    "P2": P2_CONFIG,
    "P2_keep_stopwords": P2_KEEP_STOPWORDS,
    "P2_no_lemma": P2_NO_LEMMA,
    "P2_no_hashtag_split": P2_NO_HASHTAG_SPLIT,
    "P1_plus_lemma": P1_PLUS_LEMMA,
}

demo_text = "I am not very disappointed with the delayed flights #badcustomerservice"
print("RAW:", demo_text)
for name, cfg in VARIANTS.items():
    print(f"{name:22} -> {preprocess_text(demo_text, cfg)}")

RAW: I am not very disappointed with the delayed flights #badcustomerservice
P0                     -> I am not very disappointed with the delayed flights badcustomerservice
P1                     -> i am not very disappointed with the delayed flights badcustomerservice
P2                     -> not very disappointed delay flight bad customer service
P2_keep_stopwords      -> I be not very disappointed with the delay flight bad customer service
P2_no_lemma            -> not very disappointed delayed flights bad customer service
P2_no_hashtag_split    -> not very disappointed delay flight badcustomerservice
P1_plus_lemma          -> I be not very disappointed with the delay flight badcustomerservice


## 8. Export helper

Each export is self-contained: it writes a processed CSV **and** a JSON record of the exact configuration used. Run only the export(s) you need.

In [17]:
import json
from pprint import pprint

OUTPUT_DIR = Path("/content/drive/My Drive/MSI5001 project/EDA/0.Processed-files")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def export_variant(config_name, run_config, output_filename):
    export_df = df.copy()
    export_df["processed_text"] = (
        export_df["text"].fillna("").astype(str)
        .apply(lambda text: preprocess_text(text, run_config))
    )

    output_csv = OUTPUT_DIR / output_filename
    config_json = OUTPUT_DIR / f"{config_name}_config.json"

    export_df.to_csv(output_csv, index=False)
    with open(config_json, "w") as f:
        json.dump(run_config, f, indent=2)

    print("Configuration:", config_name)
    print("Rows exported:", len(export_df))
    print("CSV saved:", output_csv)
    print("Config saved:", config_json)
    pprint(run_config)
    display(export_df[["text", "processed_text"]].head(10))
    return export_df

## 9. Export one variation at a time

These subsections are independent. P2 can take longer because spaCy lemmatization and stopword removal are enabled.

### 9.1 P0 — basic cleaning

In [18]:
p0_df = export_variant("p0", {**P0_CONFIG}, "Tweets-processed-p0.csv")

Configuration: p0
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p0.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p0_config.json
{'decode_html_entities': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'none',
 'normalize_whitespace': True,
 'remove_other_mentions': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'strip_hashtag_symbols': True}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline What said.
1,@VirginAmerica plus you've added commercials t...,airline plus you've added commercials to the e...
2,@VirginAmerica I didn't today... Must mean I n...,airline I didn't today... Must mean I need to ...
3,@VirginAmerica it's really aggressive to blast...,airline it's really aggressive to blast obnoxi...
4,@VirginAmerica and it's a really big bad thing...,airline and it's a really big bad thing about it
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously would pay $30 a flight for s...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes, nearly every time I fly VX this “..."
7,@VirginAmerica Really missed a prime opportuni...,airline Really missed a prime opportunity for ...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline Well, I didn't…but NOW I DO!:-D"
9,"@VirginAmerica it was amazing, and arrived an ...","airline it was amazing, and arrived an hour ea..."


### 9.2 P1 — text normalization

In [19]:
p1_df = export_variant("p1", {**P1_CONFIG}, "Tweets-processed-p1.csv")

Configuration: p1
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p1.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p1_config.json
{'convert_numbers': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'none',
 'lowercase': True,
 'normalize_whitespace': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'strip_hashtag_symbols': True}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline what said.
1,@VirginAmerica plus you've added commercials t...,airline plus you have added commercials to the...
2,@VirginAmerica I didn't today... Must mean I n...,airline i did not today... must mean i need to...
3,@VirginAmerica it's really aggressive to blast...,airline it is really aggressive to blast obnox...
4,@VirginAmerica and it's a really big bad thing...,airline and it is a really big bad thing about it
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously would pay $30 a flight for s...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes, nearly every time i fly vx this “..."
7,@VirginAmerica Really missed a prime opportuni...,airline really missed a prime opportunity for ...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline well, i did not…but now i do!:-d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline it was amazing, and arrived an hour ea..."


### 9.3 P2 — finalized lexical normalization

In [20]:
p2_df = export_variant("p2", {**P2_CONFIG}, "Tweets-processed-p2.csv")

Configuration: p2
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p2.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p2_config.json
{'convert_numbers': False,
 'correct_spelling': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'wordninja',
 'lemmatize': True,
 'lowercase': True,
 'max_edit_distance': 2,
 'normalize_whitespace': True,
 'preserve_negation': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_stopwords': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'segmentation_min_token_length': 7,
 'segmentation_mode': 'none',
 'spacy_model': 'en_core_web_sm',
 'strip_hashtag_symbols': True,
 'symspell_dictionary_path': None}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline .
1,@VirginAmerica plus you've added commercials t...,airline plus add commercial experience . . . t...
2,@VirginAmerica I didn't today... Must mean I n...,airline not today . . . mean need trip !
3,@VirginAmerica it's really aggressive to blast...,"airline really aggressive blast obnoxious "" en..."
4,@VirginAmerica and it's a really big bad thing...,airline really big bad thing
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously pay $ 30 flight seat not pla...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes , nearly time fly vx “ ear worm ” ..."
7,@VirginAmerica Really missed a prime opportuni...,airline really miss prime opportunity man hat ...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline , not … ! : - d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline amazing , arrive hour early . too good ."


### 9.4 P2 with stopwords kept

In [21]:
p2_keep_stopwords_df = export_variant(
    "p2_keep_stopwords",
    {**P2_KEEP_STOPWORDS},
    "Tweets-processed-p2_keep_stopwords.csv",
)

Configuration: p2_keep_stopwords
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p2_keep_stopwords.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p2_keep_stopwords_config.json
{'convert_numbers': False,
 'correct_spelling': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'wordninja',
 'lemmatize': True,
 'lowercase': True,
 'max_edit_distance': 2,
 'normalize_whitespace': True,
 'preserve_negation': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_stopwords': False,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'segmentation_min_token_length': 7,
 'segmentation_mode': 'none',
 'spacy_model': 'en_core_web_sm',
 'strip_hashtag_symbols': True,
 'symspell_dictionary_path': None}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline what say .
1,@VirginAmerica plus you've added commercials t...,airline plus you have add commercial to the ex...
2,@VirginAmerica I didn't today... Must mean I n...,airline I do not today ... must mean I need to...
3,@VirginAmerica it's really aggressive to blast...,airline it be really aggressive to blast obnox...
4,@VirginAmerica and it's a really big bad thing...,airline and it be a really big bad thing about it
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously would pay $ 30 a flight for ...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes , nearly every time I fly vx this ..."
7,@VirginAmerica Really missed a prime opportuni...,airline really miss a prime opportunity for ma...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline well , I do not … but now i do!:-d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline it be amazing , and arrive an hour ear..."


### 9.5 P2 without lemmatization

In [22]:
p2_no_lemma_df = export_variant(
    "p2_no_lemma",
    {**P2_NO_LEMMA},
    "Tweets-processed-p2_no_lemma.csv",
)

Configuration: p2_no_lemma
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p2_no_lemma.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p2_no_lemma_config.json
{'convert_numbers': False,
 'correct_spelling': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'wordninja',
 'lemmatize': False,
 'lowercase': True,
 'max_edit_distance': 2,
 'normalize_whitespace': True,
 'preserve_negation': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_stopwords': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'segmentation_min_token_length': 7,
 'segmentation_mode': 'none',
 'spacy_model': 'en_core_web_sm',
 'strip_hashtag_symbols': True,
 'symspell_dictionary_path': None}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline said .
1,@VirginAmerica plus you've added commercials t...,airline plus added commercials experience . . ...
2,@VirginAmerica I didn't today... Must mean I n...,airline not today . . . mean need trip !
3,@VirginAmerica it's really aggressive to blast...,"airline really aggressive blast obnoxious "" en..."
4,@VirginAmerica and it's a really big bad thing...,airline really big bad thing
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously pay $ 30 flight seats not pl...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes , nearly time fly vx “ ear worm ” ..."
7,@VirginAmerica Really missed a prime opportuni...,airline really missed prime opportunity men ha...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline , not … ! : - d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline amazing , arrived hour early . too good ."


### 9.6 P2 without hashtag segmentation

In [23]:
p2_no_hashtag_split_df = export_variant(
    "p2_no_hashtag_split",
    {**P2_NO_HASHTAG_SPLIT},
    "Tweets-processed-p2_no_hashtag_split.csv",
)

Configuration: p2_no_hashtag_split
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p2_no_hashtag_split.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p2_no_hashtag_split_config.json
{'convert_numbers': False,
 'correct_spelling': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'none',
 'lemmatize': True,
 'lowercase': True,
 'max_edit_distance': 2,
 'normalize_whitespace': True,
 'preserve_negation': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_stopwords': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'segmentation_min_token_length': 7,
 'segmentation_mode': 'none',
 'spacy_model': 'en_core_web_sm',
 'strip_hashtag_symbols': True,
 'symspell_dictionary_path': None}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline .
1,@VirginAmerica plus you've added commercials t...,airline plus add commercial experience . . . t...
2,@VirginAmerica I didn't today... Must mean I n...,airline not today . . . mean need trip !
3,@VirginAmerica it's really aggressive to blast...,"airline really aggressive blast obnoxious "" en..."
4,@VirginAmerica and it's a really big bad thing...,airline really big bad thing
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously pay $ 30 flight seat not pla...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes , nearly time fly vx “ ear worm ” ..."
7,@VirginAmerica Really missed a prime opportuni...,airline really miss prime opportunity man hat ...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline , not … ! : - d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline amazing , arrive hour early . too good ."


### 9.7 P1 + lemmatization only

In [24]:
p1_plus_lemma_df = export_variant(
    "p1_plus_lemma",
    {**P1_PLUS_LEMMA},
    "Tweets-processed-p1_plus_lemma.csv",
)

Configuration: p1_plus_lemma
Rows exported: 14639
CSV saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/Tweets-processed-p1_plus_lemma.csv
Config saved: /content/drive/My Drive/MSI5001 project/0.Processed-files/p1_plus_lemma_config.json
{'convert_numbers': False,
 'decode_html_entities': True,
 'emoji_mode': 'describe',
 'expand_contractions': True,
 'hashtag_segmentation_min_token_length': 7,
 'hashtag_segmentation_mode': 'none',
 'lemmatize': True,
 'lowercase': True,
 'normalize_whitespace': True,
 'punctuation_mode': 'keep',
 'remove_other_mentions': True,
 'remove_urls': True,
 'replace_airline_hashtags': True,
 'replace_airline_mentions': True,
 'replace_airline_names': True,
 'spacy_model': 'en_core_web_sm',
 'strip_hashtag_symbols': True}


,text,processed_text
0,@VirginAmerica What @dhepburn said.,airline what say .
1,@VirginAmerica plus you've added commercials t...,airline plus you have add commercial to the ex...
2,@VirginAmerica I didn't today... Must mean I n...,airline I do not today ... must mean I need to...
3,@VirginAmerica it's really aggressive to blast...,airline it be really aggressive to blast obnox...
4,@VirginAmerica and it's a really big bad thing...,airline and it be a really big bad thing about it
5,@VirginAmerica seriously would pay $30 a fligh...,airline seriously would pay $ 30 a flight for ...
6,"@VirginAmerica yes, nearly every time I fly VX...","airline yes , nearly every time I fly vx this ..."
7,@VirginAmerica Really missed a prime opportuni...,airline really miss a prime opportunity for ma...
8,"@virginamerica Well, I didn't…but NOW I DO! :-D","airline well , I do not … but now i do!:-d"
9,"@VirginAmerica it was amazing, and arrived an ...","airline it be amazing , and arrive an hour ear..."


## 10. List saved configuration records

No prior `CONFIG_NAME` variable is required. Each Section 9 export saves its own configuration automatically.

In [25]:
config_files = sorted(OUTPUT_DIR.glob("*_config.json"))
if not config_files:
    print("No saved configuration files found yet. Run any Section 9 export first.")
else:
    print("Saved configuration records:")
    for path in config_files:
        print(" -", path.name)

Saved configuration records:
 - p0_config.json
 - p1_config.json
 - p1_plus_lemma_config.json
 - p2_config.json
 - p2_keep_stopwords_config.json
 - p2_no_hashtag_split_config.json
 - p2_no_lemma_config.json


## Methodological reminders

1. Keep the raw `text` column; do not overwrite it.
2. Structural/style features and VADER should normally use raw text.
3. Sentiment labels/confidence are reserved for post-clustering evaluation only.
4. P0/P1/P2 are validated reference configurations, not proof of better clustering.
5. The configurations are tailored to the airline-tweet corpus.
6. Record the exact configuration used for every processed export.